In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
# =========================================================
# 1. STATE
# =========================================================
class State(TypedDict):
    count: int
# =========================================================
# 2. NODES
# =========================================================
def add_one(state: State):

    print("ADD_ONE running...")

    return {
        "count": state["count"] + 1
    }


def add_two(state: State):

    print("ADD_TWO running...")

    return {
        "count": state["count"] + 2
    }


# =========================================================
# 3. GRAPH
# =========================================================

builder = StateGraph(State)

builder.add_node("add_one", add_one)
builder.add_node("add_two", add_two)

builder.add_edge(START, "add_one")
builder.add_edge("add_one", "add_two")
builder.add_edge("add_two", END)


# =========================================================
# 4. CHECKPOINTER
# =========================================================

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


# =========================================================
# 5. SAME THREAD
# =========================================================

config = {
    "configurable": {
        "thread_id": "thread-001"
    }
}


# =========================================================
# 6. FIRST EXECUTION
# =========================================================

print("\n========== FIRST INVOKE ==========")

result_1 = graph.invoke(
    {
        "count": 0
    },
    config=config
)

print("Result 1:")
print(result_1)


# =========================================================
# 7. CURRENT STATE AFTER FIRST EXECUTION
# =========================================================

state_1 = graph.get_state(config)

print("\n========== STATE AFTER FIRST INVOKE ==========")

print("Thread ID:")
print(
    state_1.config["configurable"]["thread_id"]
)

print("Checkpoint ID:")
print(
    state_1.config["configurable"]["checkpoint_id"]
)

print("Values:")
print(state_1.values)


# =========================================================
# 8. SECOND EXECUTION
# =========================================================

print("\n========== SECOND INVOKE ==========")

result_2 = graph.invoke(
    {
        "count": 10
    },
    config=config
)

print("Result 2:")
print(result_2)


# =========================================================
# 9. CURRENT STATE AFTER SECOND EXECUTION
# =========================================================

state_2 = graph.get_state(config)

print("\n========== STATE AFTER SECOND INVOKE ==========")

print("Thread ID:")
print(
    state_2.config["configurable"]["thread_id"]
)

print("Checkpoint ID:")
print(
    state_2.config["configurable"]["checkpoint_id"]
)

print("Values:")
print(state_2.values)


# =========================================================
# 10. COMPLETE THREAD HISTORY
# =========================================================

print("\n========== THREAD HISTORY ==========")

history = list(
    graph.get_state_history(config)
)

for index, state in enumerate(history):

    print(f"\nCheckpoint {index}")

    print(
        "Thread ID:",
        state.config["configurable"]["thread_id"]
    )

    print(
        "Checkpoint ID:",
        state.config["configurable"]["checkpoint_id"]
    )

    print(
        "Values:",
        state.values
    )

    print(
        "Next:",
        state.next
    )


# =========================================================
# 11. FINAL OBSERVATION
# =========================================================

print("\n========================================")
print("OBSERVATION")
print("========================================")

print(
    "Same Thread ID:",
    state_1.config["configurable"]["thread_id"]
)

print(
    "Checkpoint after first invoke:",
    state_1.config["configurable"]["checkpoint_id"]
)

print(
    "Checkpoint after second invoke:",
    state_2.config["configurable"]["checkpoint_id"]
)

print(
    "\nAre Thread IDs same?",
    state_1.config["configurable"]["thread_id"]
    ==
    state_2.config["configurable"]["thread_id"]
)

print(
    "Are Checkpoint IDs different?",
    state_1.config["configurable"]["checkpoint_id"]
    !=
    state_2.config["configurable"]["checkpoint_id"]
)